# Lead Variant Effect dataset preparation

The aim of this notebook is to collect the information about the effect of the credible set lead variants.
**This includes**:

- Addition of **Major population sample size** and **size of cases/controls** from _studyIndex_,
- Addition of **VEP consequence score** derived annotations from _variantIndex_
- Addition of **study specific major ancestry variant AF** (allele frequency)<a name="out of sample AF"></a>[<sup>[1]</sup>](#cite_note-1) annotations from _variantIndex_
- Calculation of **MAF (Minor Allele Frequency)** based on AF of the **credible set lead variants** derived from _studyLocus_
- Calculation of **Variance Explained by lead variant**
- Calculation of the **Rescaled estimated effect sizes** based on the trait class (dichotomous or continuous) and the MAF of the lead variant.

<a name="cite_note-1"></a>1. [^](#cite_ref-1) AF is derived from GnomAD v4.1 allele frequencies from joint Exome and WGS datasets.


In [1]:
# Ensure proper java version < 11
!java -version


openjdk version "11.0.13" 2021-10-19
OpenJDK Runtime Environment JBR-11.0.13.7-1751.21-jcef (build 11.0.13+7-b1751.21)
OpenJDK 64-Bit Server VM JBR-11.0.13.7-1751.21-jcef (build 11.0.13+7-b1751.21, mixed mode)


## Session setup

- Create the sparkSession
- Set all input/output paths


In [33]:
import json
from pathlib import Path

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.variant_index import VariantIndex
from pyspark.sql import functions as f


In [3]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})
variant_index_path = "../../data/25.06/output/variant"
study_index_path = "../../data/25.06/output/study"
credible_set_path = "../../data/25.06/output/credible_set"


25/11/26 20:58:15 WARN Utils: Your hostname, mindos resolves to a loopback address: 127.0.1.1; using 192.168.0.100 instead (on interface eno1)
25/11/26 20:58:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/26 20:58:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [30]:
output_dataset_path = Path("../../data/intermediate_files/lead_variant_effect").as_posix()
output_dataset_schema = Path("../../src/manuscript_methods/schemas/lead_variant_effect.json").as_posix()


In [8]:
session.spark


## Building temporary dataset

The temporary dataset needs to be build from the _studyIndex_, _studyLocus_ and _variantIndex_ datasets.


In [9]:
vi = VariantIndex.from_parquet(session, variant_index_path)
si = StudyIndex.from_parquet(session, study_index_path)
cs = StudyLocus.from_parquet(session, credible_set_path)


_cs = cs.df.select(
    f.col("studyId"),
    f.col("studyLocusId"),
    f.col("variantId"),
    f.col("beta"),
    f.col("zScore"),
    f.col("pValueMantissa"),
    f.col("pValueExponent"),
    f.col("standardError"),
    f.col("finemappingMethod"),
    f.col("studyType"),
    f.col("locus"),
    f.col("isTransQtl"),
)
_si = si.df.select(
    f.col("studyId"),
    f.col("nSamples"),
    f.col("nControls"),
    f.col("nCases"),
    f.col("geneId"),  # for molqtl traits
    f.col("traitFromSourceMappedIds"),
    f.col("ldPopulationStructure"),
    f.col("traitFromSource"),
    f.col("traitFromSourceMappedIds"),
)

_vi = vi.df.select(
    f.col("variantId"),
    f.col("allelefrequencies"),
    f.col("variantEffect"),
    f.col("transcriptConsequences"),
    f.col("chromosome"),
    f.col("position"),
    f.col("referenceAllele"),
    f.col("alternateAllele"),
)

dataset = _cs.join(_si, how="left", on="studyId").join(_vi, how="left", on="variantId")


## MAF Calculation

To add the MAF (Minor Allele Frequency) to the dataset we need to extract the major ancestry from _studyIndex_ and use it to extract the relevant allele frequency from the _variantIndex_ dataset.

The MAF is calculated as follows:

<ol>
    <li>Extract the major ancestry from the <code>studyIndex</code> dataset.</li>
    <ol>
        <li>In case there are multiple ancestries that match the <code>max(relativeSampleSize)</code>, and one of them is <code>NFE</code>, use <code>NFE</code> as the major ancestry.</li>
        <li>In case there are multiple ancestries that match the <code>max(relativeSampleSize)</code> and none of them is <code>NFE</code>, use the first ancestry in the list as the major ancestry.</li>
        <li>If there is no ancestry in the list, use <code>NFE</code> as the major ancestry, assign the <code>relativeSampleSize</code> to 0.0</li>
    </ol>
    <li>Extract the allele frequency for the major ancestry from the <code>variantIndex</code> dataset.</li>
</ol>

The output of this step contains the following columns:

- `majorLdPopulation`: defined as the major ancestry from the _studyIndex_ dataset. This column contains two fields:
  - `ldPopulation`: the major ancestry value (ex. `nfe`, `afr`, etc.). In case there is no ancestry, the value is set to `nfe`. (ex. molecular QTLs).
  - `relativeSampleSize`: derived from LdPopulationStructure field. In case there is no ancestry, the value is set to `0.0`. (ex. molecular QTLs).
- `majorLdPopulationMaf`: defined as the MAF for the major ancestry from the _variantIndex_ dataset, with two fields
  - `value` represents the MAF, while
  - `type` represents weather the MAF was calculated from major ancestry AF had to be flipped or not.
- `majorLdPopulationAf`: defined as the AF for the major ancestry from the _variantIndex_ dataset. With two fields:
  - `populationName` represents the major ancestry name (ex. `nfe`, `afr`, etc.).
  - `alleleFrequency` represents the Allele Frequency for the major ancestry.
- `ldStructure`: defined as the LD structure of the populations present in the _studyIndex_ dataset:
  - `type`: type of the LD structure derived from major ancestries, ex.
    - `only_major` (one population, that is major population),
    - `one_major` (multiple populations with one major population),
    - `n_even_major` (n multiple populations, each major population has equivalent relative sample size),
    - `n_uneven_major` (n multiple populations, each major population has different relative sample size),
    - `empty_ld` (no LD structure provided - ex. molecular QTLs),
    - `no_relative_sample_size` (LD structure is provided, but without relative sample size.)
  - `majorPops`: a list of `LDPopulation` objects representing the major populations present in the LD structure. Each `LDPopulation` object contains:
    - `ldPopulation`: the ancestry value (ex. `nfe`, `afr`, etc.).
    - `relativeSampleSize`: the relative sample size for the population.

The reason for adding the `ldStructure` is to provide a more detailed information about the LD structure of major ancestries, the actual analysis makes the assumption that in the case of multiple major ancestries with even sample sizes, `nfe` is preferred over others, in case of uneven relative sample sizes, the largest major ancestry is preferred.


In [13]:
from manuscript_methods.ld_populations import LDPopulationName, LDPopulationStructure
from manuscript_methods.maf import AlleleFrequencies

ld_pop = LDPopulationStructure(f.col("ldPopulationStructure"))
major_ld_pop = ld_pop.major_population(default_major_pop=LDPopulationName.NFE)
major_ld_maf = AlleleFrequencies(f.col("alleleFrequencies")).ld_population_maf(major_ld_pop.ld_population)
major_ld_af = AlleleFrequencies(f.col("alleleFrequencies")).ld_population_af(major_ld_pop.ld_population)
ld_structure = ld_pop.ld_structure()

dataset = dataset.withColumns(
    {
        "majorLdPopulation": major_ld_pop.col,
        "majorLdPopulationMaf": major_ld_maf.col,
        "majorLdPopulationAf": major_ld_af.col,
        "majorLdPopulationsStructure": ld_structure,
    }
)

(
    dataset.select(
        "majorLdPopulation",
        "majorLdPopulationMaf",
        "majorLdPopulationAf",
        "majorLdPopulationsStructure",
    ).show(5, truncate=False)
)
(
    dataset.select(
        "majorLdPopulation",
        "majorLdPopulationMaf",
        "majorLdPopulationAf",
        "majorLdPopulationsStructure",
    ).printSchema()
)


+-----------------+----------------------------------+-------------------------------+---------------------------+
|majorLdPopulation|majorLdPopulationMaf              |majorLdPopulationAf            |majorLdPopulationsStructure|
+-----------------+----------------------------------+-------------------------------+---------------------------+
|{nfe, 1.0}       |{0.016889607526091432, notFlipped}|{nfe_adj, 0.016889607526091432}|{only_major, [{nfe, 1.0}]} |
|{nfe, 0.0}       |{0.4487081283327932, notFlipped}  |{nfe_adj, 0.4487081283327932}  |{empty_ld, []}             |
|{nfe, 0.0}       |{0.2906750992793058, flipped}     |{nfe_adj, 0.7093249007206942}  |{empty_ld, []}             |
|{nfe, 1.0}       |{0.32507018380210817, notFlipped} |{nfe_adj, 0.32507018380210817} |{only_major, [{nfe, 1.0}]} |
|{nfe, 1.0}       |{0.32507018380210817, notFlipped} |{nfe_adj, 0.32507018380210817} |{only_major, [{nfe, 1.0}]} |
+-----------------+----------------------------------+--------------------------

## Variance explained by lead variant (Approximation)

The code below is used to calculate the PVE (Phenotypic Variance Explained) by the lead variant in the credible set.

The variance explained follows the simplified formula

${variance\;explained}=\chi^2 / n $ 


- The $\chi^2$ is calculated as **Inverse survival function** by using `scipy.stats.isf` function from lead variant $pValue$ (depicted as `pValueMantissa` and `pValueExponent`).
- The $n$ parameter is the number of samples derived from GWAS study description.

- In case where the `pValueExponent < 300` to avoid floating point errors we estimate $\chi^2$ statistic with $-log_{10}(pValue)$
- The $variance\;explained$ can be only calculated where the $n > 0$

* [Rosenberg MS. A generalized formula for converting chi-square tests to effect sizes for meta-analysis. PLoS One. 2010 Apr 7;5(4):e10059. doi: 10.1371/journal.pone.0010059. PMID: 20383281; PMCID: PMC2850938.](https://pmc.ncbi.nlm.nih.gov/articles/PMC2850938/)

In [11]:
from manuscript_methods.variant_statistics import PValueComponents, VariantStatistics

pval_components = PValueComponents(p_value_mantissa=f.col("pValueMantissa"), p_value_exponent=f.col("pValueExponent"))
n_samples = f.col("nSamples")
variant_stats = VariantStatistics.compute(pval_components, n_samples)

dataset = dataset.withColumns({"variantStatistics": variant_stats.col})
dataset.select("variantStatistics").show(5, truncate=False)
dataset.select("variantStatistics").printSchema()


/home/mindos/Projects/OpenTargets/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning: In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.


+--------------------------------------------------------+
|variantStatistics                                       |
+--------------------------------------------------------+
|{50.844127911818155, 1.0, -12, 1.6738813053705747E-5}   |
|{36.00017305956795, 1.973, -9, 0.06271807153234835}     |
|{37.5342889430328, 8.982, -10, 0.07055317470494887}     |
|{32.70463990170132, 1.0728083, -8, 8.287166571652616E-5}|
|{34.90715636972957, 3.458075, -9, 8.60447645955975E-5}  |
+--------------------------------------------------------+
only showing top 5 rows

root
 |-- variantStatistics: struct (nullable = false)
 |    |-- chi2Stat: double (nullable = true)
 |    |-- pValueMantissa: float (nullable = true)
 |    |-- pValueExponent: integer (nullable = true)
 |    |-- ApproximatedVarianceExplained: double (nullable = true)



## Study statistics

The code below is used to combine and classify the cohort statistics from the _studyIndex_ dataset.

This includes:

- n_cases
- n_controls
- n_samples
- study_type
- trait
- trait_ids
- gene_id


In [12]:
from manuscript_methods.study_statistics import StudyStatistics

cohort_stat = StudyStatistics.compute(
    n_samples=f.col("nSamples"),
    n_cases=f.col("nCases"),
    n_controls=f.col("nControls"),
    trait=f.col("traitFromSource"),
    study_type=f.col("studyType"),
    is_trans_pqtl=f.col("isTransQtl"),
    gene_id=f.col("geneId"),
)

dataset = dataset.withColumns({"studyStatistics": cohort_stat.col})
dataset.select("studyStatistics").show(5, truncate=False)
dataset.select("studyStatistics").printSchema()


+-------------------------------------------------------------------------------+
|studyStatistics                                                                |
+-------------------------------------------------------------------------------+
|{0, 0, 3037499, Educational attainment, gwas, quantitative, NULL}              |
|{NULL, NULL, 574, ENST00000525249, eqtl, quantitative, ENSG00000250305}        |
|{0, 0, 357580, Glucose levels (UKB data field 30740), gwas, quantitative, NULL}|
|{0, 0, 394642, Glucose levels (UKB data field 30740), gwas, quantitative, NULL}|
|{0, 0, 405686, Random glucose levels, gwas, quantitative, NULL}                |
+-------------------------------------------------------------------------------+
only showing top 5 rows

root
 |-- studyStatistics: struct (nullable = false)
 |    |-- nCases: integer (nullable = true)
 |    |-- nControls: integer (nullable = true)
 |    |-- nSamples: integer (nullable = true)
 |    |-- trait: string (nullable = true)
 |   

## Rescaling of the marginal effect size

Due to the fact that the `beta` reported by the credible sets lead variant can be of unknown origin ($\mu$ from SuSiE or beta from summary statistics) or scale, we want to rescale the marginal effect size from the $\chi^2$ statistic.

The rescaling of the marginal effect size is done via two formulas depending on trait being **quantitative** or **binary**.

### Procedure overview

1. Compute the $\chi^2$ statistic from the $p-value$ of the lead variant.
2. Estimate the model used for beta estimation based on the trait type (binary or quantitative).
3. Compute the $z-score$ from the $\chi^2$ statistic value.
4. Compute the standard error ($se$) based on the model type and MAF of the lead variant.
5. Compute the rescaled marginal effect size.

### $\chi^2$ computation

The $\chi2^2$ is computed as the inverse survival function of the $pValue$ of the lead variant using `scipy.stats.isf` function. This step is done in the VariantStatistics paragraph.

### Trait type estimation

Estimation of the trait type is done on the basis of availability of reported `nCases` and `nControls` fields in the study description. This step is done in the StudyStatistics paragraph.

- In case both fields are non empty and non zero we assume _binary trait_
- In case cases are zero or are not reported we assume _quantitative trait_

### Calculation of the $z-score$

The $z-score$ is computed as follows:

$$z-score = \frac{\beta}{|{\beta}|} \cdot \sqrt{\chi^2}$$

Where

- $\beta$ - _standardised beta reported from in the summary statistics_
- $\frac{\beta}{|{\beta}|}$ is used to find the direction of the effect.
- In case when $\beta$ was not reported we assumed the $\frac{\beta}{|{\beta}|}$ to be unknown

### Calculation of the standard error ($se$)

In both cases we estimate the marginal effect size $estimated\;\beta$ with following formula
$$estimated\;\beta = zscore \cdot se$$

#### Binary trait marginal effect size estimation

$$se = \frac{1}{\sqrt{(varG \cdot prev \cdot (1 - prev))}}$$

- $varG = 2  \cdot f \cdot (1 - f)$ - _component of genetic variance_ - the original is $var_{G} = 2\beta^2f(1 - f)$
- $f$ - _Minor Allele Frequency of lead variant_
- $prev = \frac{nCases}{nSamples}$ - _Trait prevelance_

#### Quantative trait marginal effect size estimation

$$se = \frac{1}{\sqrt{n \cdot varG}}$$

- $varG = 2  \cdot f \cdot (1 - f)$
- $f$ - _Minor Allele Frequency of lead variant_

The $\chi^2$ was esteimated as described in `variance Explained` calculation.


In [ ]:
from manuscript_methods.maf import MinorAlleleFrequency, PopulationFrequency
from manuscript_methods.rescaled_beta import RescaledStatistics

rescaled_stats = RescaledStatistics.compute(
    beta=f.col("beta"),
    chi2_stat=VariantStatistics(f.col("variantStatistics")).chi2_stat,
    trait_class=StudyStatistics(f.col("studyStatistics")).trait_class,
    af=PopulationFrequency(f.col("majorLdPopulationAf")).allele_frequency,
    maf=MinorAlleleFrequency(f.col("majorLdPopulationMaf")).value,
    n_cases=StudyStatistics(f.col("studyStatistics")).n_cases,
    n_samples=StudyStatistics(f.col("studyStatistics")).n_samples,
)

dataset = dataset.withColumns({"rescaledStatistics": rescaled_stats.col})

dataset.select("rescaledStatistics").show(5, truncate=False)
dataset.select("rescaledStatistics").printSchema()

# Check for empty rescaled beta

# NOTE:
# Reasons for null rescaled beta could be:
# - missing beta in the summary statistics
# - af == 0.0

(
    dataset.select(
        "rescaledStatistics",
        "beta",
        f.col("majorLdPopulationMaf.value").alias("maf"),
        f.col("majorLdPopulationAf.alleleFrequency").alias("af"),
    )
    .filter(f.col("rescaledStatistics").getField("estimatedBeta").isNull())
    .select("rescaledStatistics.estimatedBeta", "beta", "maf", "af")
    .show(5, truncate=False)
)


+----------------------------------------------------------------------------------------------------------------------------+
|rescaledStatistics                                                                                                          |
+----------------------------------------------------------------------------------------------------------------------------+
|{+, 5.649089354115727, 0.41301148669347476, NULL, 0.06455514613492445, 0.3646777887841868, 0.3646777887841868}              |
|{+, 5.527028538643197, 0.41301148669347476, NULL, 0.06746260680414594, 0.37286775309777936, 0.37286775309777936}            |
|{-, -5.165341749369594, 0.07501418483343926, NULL, 0.2641871070847862, -1.3646166938702218, -1.3646166938702218}            |
|{+, 6.189122637056776, 2.9401819966301894E-5, 0.011541157705449879, 3.177487162245169, 19.66585772480887, 19.66585772480887}|
|{+, 5.240433283644419, 0.47805311731099387, NULL, 0.09599508426010712, 0.5030558346229159, 0.5030558346229159}

+-------------+------------------+-------------------+-------------------+
|estimatedBeta|beta              |maf                |af                 |
+-------------+------------------+-------------------+-------------------+
|NULL         |NULL              |0.49138642208958117|0.5086135779104188 |
|NULL         |2.3626536194601653|0.0                |0.0                |
|NULL         |NULL              |0.3931467124341599 |0.3931467124341599 |
|NULL         |NULL              |0.35020911292097734|0.35020911292097734|
|NULL         |NULL              |0.35020911292097734|0.35020911292097734|
+-------------+------------------+-------------------+-------------------+
only showing top 5 rows



## VEP consequence extraction

The VEP consequence extraction is done with two manners:

1. Looking for the **direct most severe consequence** for the lead variant. This is done by linking the lead variant to the gene with most severe consequence in within the +/- 500kb region of the lead variant. - `mostSevereConsequence`
2. Looking for the consequence linked to the molecular QTL egene
   - If the `geneId` (egene) is found in transcript annotations (**in-gene effect**),
   - If the `geneId` (egene) is not found in the transcript consequences, we expect that the gene within the +/- 500kb region of the lead variant, which has the most severe consequence to have distal effect (**out-gene effect**) on the egene. - `mostSevereConsequenceForMolecularTrait`


In [24]:
from manuscript_methods.study_statistics import StudyStatistics
from manuscript_methods.tc import LeadVariantConsequences, TranscriptConsequences

tc = TranscriptConsequences(f.col("transcriptConsequences"))
sstats = StudyStatistics(f.col("studyStatistics"))
lc = LeadVariantConsequences.compute(tc, sstats)


dataset = dataset.withColumn(lc.name, lc.col)
dataset.select(lc.name).show(5, truncate=False)
dataset.select(lc.name).printSchema()


+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|leadVariantConsequence                                                                                                                                                                                                                                                                                                                                                            |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Locus statistics

The locus statistics gets:

- Posterior Probability of the lead variant
- Locus length
- Locus size
- Locus start - (start of the first variant in the locus)
- Locus end - (end of the last variant in the locus)

All statistics are derived from the **studyLocus** dataset `locus` object.


In [25]:
from manuscript_methods.locus_statistics import LocusStatistics

ls = LocusStatistics.compute(locus=f.col("locus"), lead_variant=f.col("variantId"))
dataset = dataset.withColumn(ls.name, ls.col)
dataset.select(ls.name).show(5, truncate=False)
dataset.select(f"{ls.name}.*").printSchema()


+----------------------------------------------------+
|locusStatistics                                     |
+----------------------------------------------------+
|{27, 115202, 77713486, 77828688, 0.8890875456014443}|
|{8, 13207, 12941365, 12954572, 0.4030465170234884}  |
|{17, 80354, 34999010, 35079364, 0.07973650519241357}|
|{19, 8500, 93622662, 93631162, 0.09649171692358438} |
|{16, 6767, 93624395, 93631162, 0.08936639097922486} |
+----------------------------------------------------+
only showing top 5 rows

root
 |-- locusSize: integer (nullable = false)
 |-- locusLength: integer (nullable = true)
 |-- locusStart: integer (nullable = true)
 |-- locusEnd: integer (nullable = true)
 |-- leadVariantPIP: double (nullable = true)



## Variant type for interval joins

Compute the variant type and effective length of Indels to use them downstream for interval joins


In [26]:
from manuscript_methods.variant_type import Variant

dataset = dataset.withColumn(
    "variant",
    Variant.compute(f.col("chromosome"), f.col("position"), f.col("referenceAllele"), f.col("alternateAllele")).col,
)

dataset.select("variant").show(5, truncate=False)
dataset.select("variant").printSchema()


+--------------------------------------+
|variant                               |
+--------------------------------------+
|{2, 77742537, 77742537, SNV, A, T, 0} |
|{8, 12953425, 12953425, SNV, A, G, 0} |
|{9, 35063727, 35063727, SNV, A, G, 0} |
|{10, 93629326, 93629326, SNV, A, G, 0}|
|{10, 93629326, 93629326, SNV, A, G, 0}|
+--------------------------------------+
only showing top 5 rows

root
 |-- variant: struct (nullable = false)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: integer (nullable = true)
 |    |-- end: integer (nullable = true)
 |    |-- type: string (nullable = true)
 |    |-- ref: string (nullable = true)
 |    |-- alt: string (nullable = true)
 |    |-- length: integer (nullable = true)



## Final dataset contract


In [27]:
dataset = dataset.select(
    f.col("variantId"),
    f.col("variant"),
    f.col("studyLocusId"),
    f.col("studyId"),
    f.col("geneId"),
    f.col("beta").alias("originalBeta"),
    f.col("standardError").alias("originalStandardError"),
    f.col("locusStatistics"),
    f.col("finemappingMethod"),
    f.col("isTransQtl"),
    f.col("variantEffect"),
    f.col("majorLdPopulation"),
    f.col("majorLdPopulationMaf"),
    f.col("majorLdPopulationAf"),
    f.col("variantStatistics"),
    f.col("studyStatistics"),
    f.col("rescaledStatistics"),
    f.col("leadVariantConsequence"),
    f.col("traitFromSourceMappedIds"),
)


### Save the dataset to parquet


In [31]:
dataset.repartition(50).write.mode("overwrite").parquet(output_dataset_path)


### Save the schema 

In [34]:
with open(output_dataset_schema, "w") as fp:
    json.dump(json.loads(dataset.schema.json()), fp, indent=2)


In [35]:
print(f"Dataset with {dataset.count()} rows saved to {output_dataset_path}")


Dataset with 2833758 rows saved to ../../data/intermediate_files/lead_variant_effect
